# Generating Synthetic Data

- Write models that can generate datasets
- Use a variety of models and prompts for diverse outputs
- Create a Gradio UI for your products

## Approach
We'll build a **Synthetic Dataset Generator** that:
1. Uses **multiple LLM providers** (Groq with Llama, Google Gemini, DeepSeek via OpenRouter) for diverse outputs
2. Lets users specify the dataset topic, number of rows, and columns
3. Outputs structured CSV data
4. Provides a **Gradio UI** for easy interaction

## Setup and Imports

In [ ]:
%pip install -q groq openai gradio pandas python-dotenv

In [ ]:
import os
import re
import pandas as pd
from io import StringIO
from dotenv import load_dotenv
from groq import Groq
from openai import OpenAI
import gradio as gr

load_dotenv()

# Groq client (direct API - free tier)
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# OpenRouter client (routes to Gemini and DeepSeek)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

print("All clients initialized successfully!")

## System Prompt for Data Generation

We create a reusable system prompt that instructs the LLM to act as a synthetic data generator, returning only valid CSV.

In [ ]:
SYSTEM_PROMPT = """You are a synthetic data generator. Your ONLY job is to produce CSV data.

Rules:
1. Output ONLY valid CSV — no markdown, no code fences, no explanations
2. The first line must be the header row
3. Use commas as delimiters
4. Wrap any field containing commas in double quotes
5. Generate realistic, diverse, and varied data — avoid repetitive patterns
6. Every row must be unique
"""

def build_user_prompt(topic, num_rows, columns):
    """Build the user prompt requesting synthetic data."""
    col_spec = columns if columns.strip() else "appropriate columns for the topic"
    return (
        f"Generate exactly {num_rows} rows of synthetic data about: {topic}\n"
        f"Columns: {col_spec}\n"
        f"Remember: output ONLY the CSV with header, nothing else."
    )

## Model-Specific Generation Functions

We use three different models for diverse outputs:
- **Groq** (Llama 3.3 70B) — direct API, fast inference
- **Gemini 2.0 Flash** — via OpenRouter
- **DeepSeek V3** — via OpenRouter

In [ ]:
def generate_with_groq(user_prompt):
    """Generate data using Groq (Llama 3.3 70B) - direct API."""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=4096,
    )
    return response.choices[0].message.content.strip()


def generate_with_gemini(user_prompt):
    """Generate data using Google Gemini 2.0 Flash via OpenRouter."""
    response = openrouter_client.chat.completions.create(
        model="google/gemini-2.0-flash-001",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=4096,
    )
    return response.choices[0].message.content.strip()


def generate_with_deepseek(user_prompt):
    """Generate data using DeepSeek V3 via OpenRouter."""
    response = openrouter_client.chat.completions.create(
        model="deepseek/deepseek-chat-v3-0324",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=4096,
    )
    return response.choices[0].message.content.strip()


MODELS = {
    "Groq (Llama 3.3 70B)": generate_with_groq,
    "Gemini 2.0 Flash (OpenRouter)": generate_with_gemini,
    "DeepSeek V3 (OpenRouter)": generate_with_deepseek,
}

print(f"Registered {len(MODELS)} model backends: {', '.join(MODELS.keys())}")

## CSV Parsing and Cleaning

LLMs sometimes wrap output in markdown code fences. We clean that up and parse into a DataFrame.

In [ ]:
import re

def clean_csv_response(raw_text):
    """Strip markdown code fences and extra whitespace from LLM output."""
    text = raw_text.strip()
    # Remove ```csv ... ``` or ``` ... ``` wrappers
    text = re.sub(r"^```(?:csv)?\s*\n?", "", text)
    text = re.sub(r"\n?```\s*$", "", text)
    return text.strip()


def parse_csv(raw_text):
    """Clean LLM output and parse it into a pandas DataFrame."""
    cleaned = clean_csv_response(raw_text)
    df = pd.read_csv(StringIO(cleaned))
    return df


def generate_dataset(topic, num_rows, columns, model_name):
    """Main generation function: calls the selected model and returns a DataFrame + raw CSV."""
    user_prompt = build_user_prompt(topic, num_rows, columns)
    generate_fn = MODELS[model_name]

    raw_csv = generate_fn(user_prompt)
    df = parse_csv(raw_csv)

    return df, clean_csv_response(raw_csv)

## Quick Test

Let's test each model with a small dataset before building the UI.

In [ ]:
# Test all three models with the same prompt
test_topic = "online bookstore customers"
test_columns = "customer_id, name, email, favorite_genre, books_purchased, membership_tier"

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)
    try:
        df_test, csv_test = generate_dataset(test_topic, 5, test_columns, model_name)
        print(f"Generated {len(df_test)} rows with columns: {list(df_test.columns)}")
        display(df_test)
    except Exception as e:
        print(f"Error: {e}")

## Gradio UI

The UI lets users:
- Enter a dataset topic and optionally specify columns
- Choose the number of rows (5–50)
- Select which model to use
- View the generated data as a table and download it as CSV

In [ ]:
def gradio_generate(topic, num_rows, columns, model_name):
    """Wrapper for Gradio: generates data and returns table + CSV text + downloadable file."""
    if not topic.strip():
        return pd.DataFrame(), "Please enter a dataset topic.", None

    try:
        df, raw_csv = generate_dataset(topic, int(num_rows), columns, model_name)

        # Save to a temp CSV file for download
        csv_path = "generated_dataset.csv"
        df.to_csv(csv_path, index=False)

        return df, raw_csv, csv_path
    except Exception as e:
        return pd.DataFrame(), f"Error: {str(e)}", None


with gr.Blocks(title="Synthetic Dataset Generator") as app:
    gr.Markdown("# Synthetic Dataset Generator")
    gr.Markdown("Generate realistic synthetic datasets using multiple LLM providers.")

    with gr.Row():
        with gr.Column(scale=2):
            topic_input = gr.Textbox(
                label="Dataset Topic",
                placeholder="e.g., hospital patients, e-commerce transactions, weather stations...",
            )
            columns_input = gr.Textbox(
                label="Columns (optional)",
                placeholder="e.g., id, name, age, city, score — leave blank for auto",
            )
        with gr.Column(scale=1):
            num_rows_slider = gr.Slider(
                minimum=5, maximum=50, step=5, value=10, label="Number of Rows"
            )
            model_dropdown = gr.Dropdown(
                choices=list(MODELS.keys()),
                value=list(MODELS.keys())[0],
                label="Model",
            )
            generate_btn = gr.Button("Generate Dataset", variant="primary")

    gr.Markdown("---")

    dataframe_output = gr.Dataframe(label="Generated Dataset", wrap=True)
    raw_csv_output = gr.Textbox(label="Raw CSV", lines=8, interactive=False)
    download_output = gr.File(label="Download CSV")

    generate_btn.click(
        fn=gradio_generate,
        inputs=[topic_input, num_rows_slider, columns_input, model_dropdown],
        outputs=[dataframe_output, raw_csv_output, download_output],
    )

    gr.Markdown("---")
    gr.Markdown(
        "**Models:** Groq (Llama 3.3 70B) | Gemini 2.0 Flash (OpenRouter) | DeepSeek V3 (OpenRouter)"
    )

app.launch()